<a href="https://colab.research.google.com/github/udlbook/udlbook/blob/main/Notebooks/Chap13/13_4_Graph_Attention_Networks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Notebook 13.4: Graph attention networks**

This notebook builds a graph attention mechanism from scratch, as discussed in section 13.8.6 of the book and illustrated in figure 13.12c

Work through the cells below, running each cell in turn. In various places you will see the words "TODO". Follow the instructions at these places and make predictions about what is going to happen or write code to complete the functions.

Contact me at udlbookmail@gmail.com if you find any mistakes or have any suggestions.



In [ ]:
import numpy as np
import matplotlib.pyplot as plt

The self-attention mechanism maps $N$ inputs $\mathbf{x}_{n}\in\mathbb{R}^{D}$ and returns $N$ outputs $\mathbf{x}'_{n}\in \mathbb{R}^{D}$.  



In [ ]:
# Set seed so we get the same random numbers
np.random.seed(1)
# Number of nodes in the graph
N = 8
# Number of dimensions of each input
D = 4

# Define a graph
A = np.array([[0,1,0,1,0,0,0,0],
              [1,0,1,1,1,0,0,0],
              [0,1,0,0,1,0,0,0],
              [1,1,0,0,1,0,0,0],
              [0,1,1,1,0,1,0,1],
              [0,0,0,0,1,0,1,1],
              [0,0,0,0,0,1,0,0],
              [0,0,0,0,1,1,0,0]]);
print(A)

# Let's also define some random data
X = np.random.normal(size=(D,N))

We'll also need the weights and biases for the keys, queries, and values (equations 12.2 and 12.4)

In [ ]:
# Choose random values for the parameters
omega = np.random.normal(size=(D,D))
beta = np.random.normal(size=(D,1))
phi = np.random.normal(size=(2*D,1))

We'll need a softmax operation that operates on the columns of the matrix and a ReLU function as well

In [ ]:
# Define softmax operation that works independently on each column
def softmax_cols(data_in):
  # Exponentiate all of the values
  exp_values = np.exp(data_in - np.max(data_in, axis=0, keepdims=True))
  # Sum over columns
  denom = np.sum(exp_values, axis = 0);
  # Replicate denominator to N rows
  denom = np.matmul(np.ones((data_in.shape[0],1)), denom[np.newaxis,:])
  # Compute softmax
  softmax = exp_values / denom
  # return the answer
  return softmax


# Define the Rectified Linear Unit (ReLU) function
def ReLU(preactivation):
  activation = preactivation.clip(0.0)
  return activation


In [ ]:
 # Now let's compute self attention in matrix form
def graph_attention(X, omega, beta, phi, A):
  D, N = X.shape
  
  # 1. Compute X_prime: linear transformation of features
  # X_prime = Omega * X + beta
  X_prime = omega @ X + beta

  # 2. Compute S: the pre-softmax attention scores
  # We need a score for every pair (n, m). 
  # S[n, m] = phi^T [x'_n; x'_m]
  S = np.zeros((N, N))
  phi_left = phi[:D, :]
  phi_right = phi[D:, :]
  
  # This is equivalent to broadcasting the score components
  scores_left = phi_left.T @ X_prime   # 1 x N
  scores_right = phi_right.T @ X_prime # 1 x N
  S = scores_left.T + scores_right     # N x N

  # 3. Apply the mask
  # Mask includes self-connections (A+I)
  mask = A + np.eye(N)
  S[mask == 0] = -1e20

  # 4. Run the softmax function to compute the attention values
  # Note: softmax_cols calculates probabilities for incoming connections to each node m
  # To match standard GAT (where neighbors of n are averaged), we softmax over rows
  # But based on common book notation where columns are nodes, we softmax over columns of S
  attention = softmax_cols(S)

  # 5. Postmultiply X' by the attention values
  # The output at node n is a weighted sum of its neighbors' features
  output = X_prime @ attention

  # 6. Apply the ReLU function
  output = ReLU(output)

  return output;

In [ ]:
# Test out the graph attention mechanism
np.set_printoptions(precision=3, suppress=True)
output = graph_attention(X, omega, beta, phi, A);
print("Correct answer is:")
print("[[0.    0.028 0.37  0.    0.97  0.    0.    0.698]")
print(" [0.    0.    0.    0.    1.184 0.    2.654 0.   ]")
print(" [1.13  0.564 0.    1.298 0.268 0.    0.    0.779]")
print(" [0.825 0.    0.    1.175 0.    0.    0.    0.   ]]")


print("\nYour answer is:")
print(output)

### **Dot-Product Graph Attention (TODO Implementation)**
Combining dot-product attention from Practical 12.1 with the graph geometry (masking).

In [ ]:
def dot_product_graph_attention(X, A):
    # Simple dot product attention without learned weights for demonstration
    # S = X^T * X
    S = X.T @ X
    
    # Apply graph mask (A+I)
    mask = A + np.eye(X.shape[1])
    S[mask == 0] = -1e20
    
    # Normalize over neighbors
    attention = softmax_cols(S)
    
    # Aggregate
    return X @ attention

output_dot = dot_product_graph_attention(X, A)
print("Dot-product graph attention output shape:", output_dot.shape)